# Quantum State & Fidelity Estimation Pipeline

This notebook runs the complete end-to-end training and evaluation pipeline using the **Quantum-Inspired Reservoir (QIR)** model to estimate quantum channel entanglement fidelity under dynamical phase and amplitude damping noise.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Generated Dataset
try:
    df = pd.read_csv('../data/quantum_channel_noise.csv')
except FileNotFoundError:
    # Fallback inline data generation if CSV is missing
    t = np.linspace(0, 20, 2000)
    gamma_phase = 0.5 * np.sin(0.1 * t) + 0.3 * np.cos(0.05 * t) + 0.1 * np.random.normal(size=2000)
    gamma_amp = 0.4 * np.cos(0.08 * t) + 0.2 * np.sin(0.15 * t) + 0.08 * np.random.normal(size=2000)
    fidelity = np.clip(0.9 * np.exp(-0.2 * (np.abs(gamma_phase) + np.abs(gamma_amp))) + 0.02 * np.random.normal(size=2000), 0, 1)
    df = pd.DataFrame({'time': t, 'gamma_phase': gamma_phase, 'gamma_amp': gamma_amp, 'fidelity': fidelity})

# 2. Preprocessing
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_raw = df[['gamma_phase', 'gamma_amp']].values
y_raw = df['fidelity'].values.reshape(-1, 1)

X_scaled = scaler_x.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw)

# Sequence generation
def create_sequences(X, y, seq_len=30):
    xs, ys = [], []
    for i in range(len(X) - seq_len):
        xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return torch.tensor(np.array(xs), dtype=torch.float32), torch.tensor(np.array(ys), dtype=torch.float32)

seq_len = 30
X_seq, y_seq = create_sequences(X_scaled, y_scaled, seq_len)

train_size = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

# 3. Quantum-Inspired Reservoir Model Definition
class QIRPredictor(nn.Module):
    def __init__(self, input_dim=2, reservoir_dim=128, spectral_radius=0.95, leak_rate=0.3):
        super(QIRPredictor, self).__init__()
        self.reservoir_dim = reservoir_dim
        self.leak_rate = leak_rate
        
        self.W_in = nn.Parameter(torch.FloatTensor(reservoir_dim, input_dim).uniform_(-0.1, 0.1), requires_grad=False)
        U = scipy.stats.ortho_group.rvs(dim=reservoir_dim)
        W_res = torch.tensor(U, dtype=torch.float32)
        max_eig = torch.max(torch.abs(torch.linalg.eigvals(W_res)))
        self.W_res = nn.Parameter((W_res / max_eig) * spectral_radius, requires_grad=False)
        
        self.readout = nn.Linear(reservoir_dim, 1)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        h_prev = torch.zeros(batch_size, self.reservoir_dim, device=x.device)
        
        for t in range(seq_len):
            x_t = x[:, t, :]
            pre_act = torch.matmul(x_t, self.W_in.T) + torch.matmul(h_prev, self.W_res.T)
            h_prev = (1 - self.leak_rate) * h_prev + self.leak_rate * torch.tanh(pre_act)
            
        return self.readout(h_prev)

# 4. Model Training
model = QIRPredictor(input_dim=2, reservoir_dim=128)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.readout.parameters(), lr=0.01)

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train)
    loss = criterion(predictions, y_train)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

# 5. Evaluation & Visualization
model.eval()
with torch.no_grad():
    test_preds = model(X_test).numpy()
    y_test_np = y_test.numpy()

test_preds_orig = scaler_y.inverse_transform(test_preds)
y_test_orig = scaler_y.inverse_transform(y_test_np)

mse = mean_squared_error(y_test_orig, test_preds_orig)
r2 = r2_score(y_test_orig, test_preds_orig)

print(f"\n[+] Test MSE: {mse:.6f}")
print(f"[+] Test R2 Score: {r2:.4f}")

plt.figure(figsize=(12, 5))
plt.plot(y_test_orig, label='True Fidelity', color='blue', alpha=0.8)
plt.plot(test_preds_orig, label='QIR Estimated Fidelity', color='red', linestyle='--', alpha=0.8)
plt.title('Quantum Channel Entanglement Fidelity Estimation using QIR')
plt.xlabel('Time Step')
plt.ylabel('Fidelity')
plt.legend()
plt.grid(True)
plt.show()